In [ ]:
# The model architecture, drawn with matplotlib (run this cell to see it).
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_architecture(attn_label):
    blocks = [
        ("input tokens   e.g. [the, movie, was, really, great, long, story]", "#e8e8e8"),
        ("token embedding  +  positional embedding", "#cfe8ff"),
        (attn_label + "\nsoftmax(Q Ka / sqrt(d)) . V   <- attention weights we visualize", "#ffd9b3"),
        ("add & LayerNorm  (residual)", "#e8e8e8"),
        ("feed-forward   Linear -> ReLU -> Linear", "#d6f5d6"),
        ("add & LayerNorm  (residual)", "#e8e8e8"),
        ("mean-pool over sequence", "#cfe8ff"),
        ("linear classifier -> 2 logits  (NEGATIVE / POSITIVE)", "#f5cccc"),
    ]
    fig, ax = plt.subplots(figsize=(8, 9))
    n = len(blocks); h, gap = 0.9, 0.35; y = n * (h + gap)
    for i, (text, color) in enumerate(blocks):
        y -= (h + gap)
        ax.add_patch(FancyBboxPatch((0.5, y), 7, h, boxstyle="round,pad=0.02,rounding_size=0.1",
                                    linewidth=1.5, edgecolor="#333333", facecolor=color))
        ax.text(4, y + h / 2, text, ha="center", va="center", fontsize=10)
        if i > 0:
            ax.add_patch(FancyArrowPatch((4, y + h + gap), (4, y + h),
                                         arrowstyle="-|>", mutation_scale=16, color="#333333"))
    ax.set_xlim(0, 8); ax.set_ylim(0, n * (h + gap)); ax.axis("off")
    ax.set_title("Model architecture", fontsize=13, weight="bold")
    plt.tight_layout(); plt.show()

draw_architecture("single-head self-attention")

# A simple Transformer — and looking at its attention

Built by hand in plain PyTorch (no `transformers`, no `nn.MultiheadAttention`) on a tiny synthetic *keyword-sentiment* dataset, so we can read the attention weights out and plot them.

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# same three-way device selection as 04_cnn_pytorch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
# A tiny toy vocabulary. Every "sentence" is a fixed-length list of words filled
# with neutral words plus exactly ONE sentiment keyword that decides the label.
# The transformer has to learn to *look at* that one word to classify the sentence.

filler_words = ["the", "movie", "was", "a", "film", "and", "quite",
                "very", "really", "long", "short", "story", "i", "think", "it"]
positive_words = ["great", "good", "amazing", "love", "brilliant"]
negative_words = ["awful", "bad", "boring", "hate", "terrible"]

vocab = filler_words + positive_words + negative_words
word2id = {w: i for i, w in enumerate(vocab)}
id2word = {i: w for w, i in word2id.items()}
vocab_size = len(vocab)

L = 7  # every sentence is exactly L words long

def make_sentence(rng):
    words = list(rng.choice(filler_words, size=L, replace=True))   # start all-neutral
    pos = rng.integers(0, L)                                       # slot for the keyword
    is_positive = rng.random() < 0.5
    keyword = rng.choice(positive_words if is_positive else negative_words)
    words[pos] = keyword
    ids = np.array([word2id[w] for w in words], dtype=np.int64)
    label = np.array([0.0, 1.0] if is_positive else [1.0, 0.0], dtype=np.float32)  # one-hot [neg, pos]
    return ids, label

class SentimentDataset(Dataset):
    def __init__(self, n, seed):
        rng = np.random.default_rng(seed)
        self.X, self.y = [], []
        for _ in range(n):
            ids, label = make_sentence(rng)
            self.X.append(ids); self.y.append(label)
        self.X = np.stack(self.X); self.y = np.stack(self.y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])

training_data = SentimentDataset(4000, seed=0)
test_data = SentimentDataset(1000, seed=1)
train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

# peek at a few generated sentences
for i in range(5):
    ids, label = training_data[i]
    words = [id2word[int(t)] for t in ids]
    print(("POS" if label.argmax() == 1 else "NEG"), " ".join(words))

In [ ]:
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, seq_len, d_model=32, d_ff=64):
        super(SimpleTransformer, self).__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)      # learned positional encoding
        # (sinusoidal alternative: precompute sin/cos of the positions instead of learning them)

        # ONE attention head, written out by hand -> we can read the weights out and plot them
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.norm1 = nn.LayerNorm(d_model)

        # position-wise feed-forward
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm2 = nn.LayerNorm(d_model)

        self.classifier = nn.Linear(d_model, 2)
        self.last_attn = None    # we stash the attention weights here for visualization

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)          # [1, T]
        h = self.token_emb(x) + self.pos_emb(pos)                    # [B, T, d]  word + position

        Q, K, V = self.Wq(h), self.Wk(h), self.Wv(h)                 # [B, T, d]
        scores = Q @ K.transpose(-2, -1) / (self.d_model ** 0.5)     # [B, T, T] similarity of every pair
        attn = torch.softmax(scores, dim=-1)                        # -> attention weights (rows sum to 1)
        self.last_attn = attn.detach()
        h = self.norm1(h + attn @ V)                                # mix values + residual + layernorm
        h = self.norm2(h + self.ff(h))                             # feed-forward + residual + layernorm

        pooled = h.mean(dim=1)                                      # average over the sequence
        return self.classifier(pooled)                             # logits [B, 2]

In [ ]:
model = SimpleTransformer(vocab_size, L).to(device)
# optimizer = torch.optim.SGD(model.parameters(), lr=.01)   # course default; too slow for a transformer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

def reset_model():
    global model, optimizer
    model = SimpleTransformer(vocab_size, L).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_one_epoch(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch_id, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if batch_id % 100 == 0:
            print(f"({batch_id}) loss: {loss:>7}")

In [ ]:
def test_one_epoch(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred_y = model(X)
            test_loss += loss_fn(pred_y, y).item()
            correct += (pred_y.argmax(1) == y.argmax(1)).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    return test_loss, correct

In [ ]:
reset_model()
for i in range(15):
    train_one_epoch(train_dataloader, model, criterion, optimizer)
    print(test_one_epoch(test_dataloader, model, criterion))

(0.005560042831348255, 1.0)
(0) loss: 0.005173700395971537


(0.0020130884731770493, 1.0)
(0) loss: 0.001891694962978363


(0.0011663701698125806, 1.0)
(0) loss: 0.001067605335265398


(0.0007773417091812007, 1.0)
(0) loss: 0.000694060348905623


(0.0005623755205306225, 1.0)
(0) loss: 0.0004961244412697852


...

(0.00023496876292483648, 1.0)
(0) loss: 0.00020862044766545296
(0.00020045811197633157, 1.0)
(0) loss: 0.0001789986272342503


(0.00017353074053971795, 1.0)
(0) loss: 0.0001559235097374767


(0.00015202599388430826, 1.0)
(0) loss: 0.00013747558114118874


(0.0001344979887107911, 1.0)
(0) loss: 0.00012239064380992204


(0.00011999556363662123, 1.0)


In [ ]:
def encode(text):
    words = text.split()
    assert len(words) == L, f"need exactly {L} words, got {len(words)}"
    ids = [word2id[w] for w in words]
    return torch.tensor([ids], device=device)      # a batch of one sentence

sentence = "the movie was really great long story"
x = encode(sentence)
model.eval()
with torch.no_grad():
    logits = model(x)
pred = "POSITIVE" if logits.argmax(1).item() == 1 else "NEGATIVE"
print(sentence, "->", pred, logits.softmax(1).cpu().numpy())

In [ ]:
def show_attention(text):
    x = encode(text)
    model.eval()
    with torch.no_grad():
        logits = model(x)
    pred = "POSITIVE" if logits.argmax(1).item() == 1 else "NEGATIVE"
    attn = model.last_attn[0].cpu().numpy()             # [L, L] attention matrix
    tokens = text.split()

    f, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(attn, cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(L)); ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticks(range(L)); ax.set_yticklabels(tokens)
    ax.set_xlabel("attended-to word (key)"); ax.set_ylabel("query word")
    ax.set_title(f"'{text}'\nprediction: {pred}")
    f.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()

# almost every word learns to attend to the single sentiment keyword -> a bright column
show_attention("the movie was really great long story")
show_attention("the movie was really boring long story")
print("done")